In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

# 1. CARGA DE DATOS
print("--- Generando Dashboard Visual ---")
df = pd.read_csv('rfm_labeled.csv')

# ==========================================
# 2. AUTO-ETIQUETADO INTELIGENTE (BUSINESS LOGIC)
# ==========================================
# Calculamos los promedios de cada cluster para saber quién es quién
perfiles = df.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()

def asignar_nombre_negocio(cluster_id):
    """Asigna nombres de negocio basándose en las reglas del RFM"""
    p = perfiles.loc[cluster_id]
    
    # Lógica:
    # Si tiene el mayor dinero -> VIP
    # Si tiene la mayor recencia (días sin venir) -> En Riesgo
    # Si tiene frecuencia baja y recencia baja -> Nuevos
    # El resto -> Leales
    
    if p['Monetary'] == perfiles['Monetary'].max():
        return 'Champions (VIP)'
    elif p['Recency'] == perfiles['Recency'].max():
        return 'En Riesgo / Dormidos'
    elif p['Frequency'] == perfiles['Frequency'].min() and p['Recency'] < perfiles['Recency'].mean():
        return 'Nuevos / Prometedores'
    else:
        return 'Clientes Leales'

# Aplicamos la función
df['Segmento'] = df['Cluster'].apply(asignar_nombre_negocio)
print("--- Distribución de Segmentos ---")
print(df['Segmento'].value_counts())

# ==========================================
# 3. GRÁFICO 1: RADAR CHART (ADN DEL CLIENTE)
# ==========================================
# Normalizamos (0 a 1) para que el gráfico se vea bien
scaler = MinMaxScaler()
df_norm = pd.DataFrame(scaler.fit_transform(df[['Recency', 'Frequency', 'Monetary']]), 
                       columns=['Recency', 'Frequency', 'Monetary'])
df_norm['Segmento'] = df['Segmento']
promedios_norm = df_norm.groupby('Segmento').mean().reset_index()

fig_radar = go.Figure()
categories = ['Recency', 'Frequency', 'Monetary']

for i, row in promedios_norm.iterrows():
    fig_radar.add_trace(go.Scatterpolar(
        r=[row['Recency'], row['Frequency'], row['Monetary']],
        theta=categories,
        fill='toself',
        name=row['Segmento']
    ))

fig_radar.update_layout(title="ADN de los Segmentos (Comparativa)", polar=dict(radialaxis=dict(visible=True, range=[0, 1])))
fig_radar.show()

# ==========================================
# 4. GRÁFICO 2: TREEMAP (¿DÓNDE ESTÁ EL DINERO?)
# ==========================================
# Agrupamos para ver: Tamaño del cuadro = Cantidad de Clientes | Color = Dinero
tree_data = df.groupby('Segmento').agg({
    'CustomerID': 'count', 
    'Monetary': 'sum'
}).reset_index()

fig_tree = px.treemap(tree_data, 
                      path=['Segmento'], 
                      values='CustomerID',
                      color='Monetary',
                      color_continuous_scale='RdBu',
                      title='Mapa de Valor: Tamaño del Segmento vs. Rentabilidad')
fig_tree.show()

# ==========================================
# 5. GRÁFICO 3: SCATTER 3D (EL UNIVERSO DE DATOS)
# ==========================================
fig_3d = px.scatter_3d(df, 
                       x='Recency', y='Frequency', z='Monetary',
                       color='Segmento', 
                       opacity=0.7, 
                       size_max=10,
                       hover_data=['CustomerID'],
                       title='Exploración 3D de Clientes (Usa el mouse para rotar)')
fig_3d.show()

--- Generando Dashboard Visual ---
--- Distribución de Segmentos ---
Segmento
Clientes Leales         375
Champions (VIP)         139
En Riesgo / Dormidos    136
Name: count, dtype: int64
